In [102]:
import sys
import pathlib
import torch
import torch.nn.functional as F
import scipy.stats as stats

PROJECT_PATH = pathlib.Path.cwd().parent
if str(PROJECT_PATH) not in sys.path:
    sys.path.append(str(PROJECT_PATH))

from pcdvq import (
    e8_minimal_directions,
    construct_direction_codebook,
    construct_magnitude_codebook,
    PCDVQ,
)
from pcdvq.utils import reshape_pq_to_k, reshape_k_to_pq

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [103]:
tensors_dumps_dir = "tensors_dumps"
weight_filename = "weight.pt"
weight_path = PROJECT_PATH / tensors_dumps_dir / weight_filename
weight = torch.load(weight_path)
p, q  = weight.shape
weight

tensor([[-2.1744e-04, -1.0193e-02,  9.5825e-03,  ..., -3.1738e-03,
         -8.6670e-03,  1.3504e-03],
        [ 2.8076e-02, -2.3804e-03, -1.2634e-02,  ..., -1.9989e-03,
          1.9684e-03,  1.5503e-02],
        [-1.6357e-02,  5.8289e-03, -1.7456e-02,  ..., -1.1414e-02,
          1.6174e-03,  2.8372e-05],
        ...,
        [ 2.2949e-02, -1.2024e-02, -2.3438e-02,  ..., -2.0752e-02,
          3.9795e-02,  2.7588e-02],
        [-2.9663e-02,  2.0142e-02,  1.4465e-02,  ..., -2.7222e-02,
         -6.2561e-03, -3.8818e-02],
        [ 1.1169e-02,  4.4922e-02, -7.2937e-03,  ...,  3.0518e-02,
         -2.3438e-02, -2.3071e-02]], dtype=torch.float16)

In [153]:
phi_bits = 6
r_bits = 2
k = 16
tau = 1e-8
tol = 1e-5
iters = 100

e8_dirs = e8_minimal_directions()
C_phi = construct_direction_codebook(e8_dirs, phi_bits)
C_r = construct_magnitude_codebook(r_bits, k, tau, tol, iters)
pcdvq = PCDVQ(directions_codebook=C_phi, magnitudes_codebook=C_r)

cur=0.06898544125075717
cur=0.6201868749804512
cur=0.8350272004099673
-----------------
cur=0.0011731041182460446
cur=0.6822845495088457
cur=0.839961517141108
-----------------
cur=0.001019614181346593
cur=0.7143606829943224
cur=0.8449116503996675
-----------------
cur=0.002197020319261787
cur=0.7319118757533091
cur=0.8485854230022644
-----------------
cur=0.003387509161167297
cur=0.7419386475283186
cur=0.8510341909102396
-----------------
cur=0.0043639118844983355
cur=0.747822916333498
cur=0.8525970946820737
-----------------
cur=0.005083714700827296
cur=0.7513326984296047
cur=0.8535749704292968
-----------------
cur=0.0055807157979329254
cur=0.7534467686705885
cur=0.8541807056298025
-----------------
cur=0.0059095725289654445
cur=0.7547275791657719
cur=0.8545538615017625
-----------------
cur=0.006121139372628995
cur=0.7555064908806902
cur=0.8547830808472873
-----------------
cur=0.006254779912880131
cur=0.7559811357811692
cur=0.8549236113155859
-----------------
cur=0.00633817703858

In [155]:
weight_reshaped  = reshape_pq_to_k(weight, k)
pcdvq_res = pcdvq.forward(weight_reshaped)
weight_reshaped_quant = pcdvq_res["x_q"]
weight_quant = reshape_k_to_pq(weight_reshaped_quant, p, q)

mse_val = F.mse_loss(weight.float(), weight_quant.float()).item()
mse_val

0.0012548166560009122

In [162]:
pcdvq_res

{'phis': tensor([[1.5781, 1.9414, 1.1973,  ..., 1.1113, 1.9121, 3.3438],
         [1.8125, 1.9980, 1.8740,  ..., 2.6113, 1.9053, 3.7559],
         [1.9814, 1.7012, 1.3672,  ..., 0.9258, 1.1748, 1.1641],
         ...,
         [1.6719, 1.6934, 1.4004,  ..., 1.4873, 2.4492, 1.1094],
         [1.1992, 1.7178, 0.7837,  ..., 1.4492, 2.2715, 2.1953],
         [1.6025, 1.3008, 1.9297,  ..., 1.6621, 0.8228, 3.9180]],
        dtype=torch.float16),
 'r': tensor([0.0282, 0.0326, 0.0287,  ..., 0.0795, 0.0868, 0.1025],
        dtype=torch.float16),
 'idx_dir': tensor([1, 1, 1,  ..., 1, 1, 1]),
 'idx_rad': tensor([1, 1, 1,  ..., 0, 0, 0]),
 'phis_q': tensor([[0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         ...,
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000,